In [38]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator
import os

load_dotenv()

True

In [31]:
model = ChatGroq(model='openai/gpt-oss-120b', api_key=os.getenv('GROQ_API_KEY'))

In [32]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback of the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [33]:
structure_model = model.with_structured_output(EvaluationSchema)

In [34]:
essay = """
Impact of Artificial Intelligence on Humans

Artificial Intelligence (AI) is rapidly transforming human life in many ways. It has improved efficiency and productivity by automating tasks, allowing people to focus on more creative and complex work. However, it also raises concerns about job loss as machines replace human labor in many industries.

In healthcare, AI helps in early diagnosis and better treatment, improving patient outcomes. In education, it enables personalized learning, making education more accessible. At the same time, excessive reliance on AI may reduce human interaction and critical thinking skills.

AI also influences daily life through smart devices and online platforms, shaping human decisions and behavior. While this increases convenience, it can create dependency and limit independent thinking.

Moreover, AI brings ethical challenges such as data privacy, bias in algorithms, and misuse of technology. These issues must be addressed to ensure fairness and security.

In conclusion, AI has both positive and negative impacts on humans. Its benefits are significant, but responsible use and proper regulation are necessary to ensure it supports human development rather than harming it.
"""

In [ ]:
prompt = f"""
Evaluate the language quality of the following essay and provide a feedback and a score.

IMPORTANT:
- Score must be an INTEGER (no decimals)
- Range: 0 to 10

Essay:
{essay}
"""
structure_model.invoke(prompt).feedback

'The essay is well‑structured and easy to follow, with a clear introduction, body paragraphs that each focus on a different aspect of AI’s impact, and a concise conclusion. The language is generally grammatical and the sentences are coherent. Vocabulary is adequate for the topic, and the writer uses appropriate transition words (e.g., "however," "at the same time," "moreover").\n\nHowever, the writing remains fairly generic and surface‑level. Many statements are broad and lack specific examples or evidence that would strengthen the argument (e.g., "AI helps in early diagnosis" without naming a technology). Some ideas are repeated (the trade‑off between convenience and dependency is mentioned twice). The essay could benefit from more varied sentence structures and richer lexical choices to avoid a monotonous tone. Additionally, a few minor stylistic issues appear, such as the phrase "its benefits are significant, but responsible use and proper regulation are necessary" which could be ti

In [ ]:
# define the state
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add] # This field will store a list of integers, and when multiple values come in, combine them using operator.add.
    average_score: float
    
    # Annotated -> Annotated[TYPE, METADATA] used to attach extra metadata to a type

In [ ]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"""
    Evaluate the language quality of the following essay and provide a feedback and a score.

    IMPORTANT:
    - Score must be an INTEGER (no decimals)
    - Range: 0 to 10

    Essay:
    {essay}
    """
    output = structure_model.invoke(prompt)
    
    return {'language_feedback': output.feedback, 'individual_score': [output.score]}

In [ ]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"""
    Evaluate the language quality of the following essay and provide a feedback and a score.

    IMPORTANT:
    - Score must be an INTEGER (no decimals)
    - Range: 0 to 10

    Essay:
    {essay}
    """
    output = structure_model.invoke(prompt)
    
    return {'language_feedback': output.feedback, 'individual_score': [output.score]}

In [46]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f"""
    Evaluate the depth of analysis of the following essay and provide a feedback and a score.

    IMPORTANT:
    - Score must be an INTEGER (no decimals)
    - Range: 0 to 10

    Essay:
    {essay}
    """
    output = structure_model.invoke(prompt)
    
    return {'analysis_feedback': output.feedback, 'individual_score': [output.score]}

In [43]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f"""
    Evaluate the clarity of thought of the following essay and provide a feedback and a score.

    IMPORTANT:
    - Score must be an INTEGER (no decimals)
    - Range: 0 to 10

    Essay:
    {essay}
    """
    output = structure_model.invoke(prompt)
    
    return {'clarity_feedback': output.feedback, 'individual_score': [output.score]}

In [44]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    
    overall_feedback = model.invoke(prompt).content
    
    average_score = sum(state["individual_score"]) / len(state["individual_score"])
    
    return {'overall_feedback': overall_feedback, 'average_score': average_score}

In [48]:
# define the graph for the state
graph = StateGraph(UPSCState)

# add node
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# add edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)


# compile the graph
workflow = graph.compile()

In [50]:
initial_state = {
    'essay': essay
}

final_state = workflow.invoke(initial_state)

print(final_state['overall_feedback'])

**Summarized Feedback**

**Strengths**
- **Clear structure & organization** – Introduction, body paragraphs (each covering a distinct domain), and conclusion are well‑sequenced, making the essay easy to follow.  
- **Appropriate vocabulary** – Key terms such as “automation,” “personalized learning,” and “bias” are used correctly for a general audience.  
- **Readability** – Simple, mostly correct sentence constructions aid comprehension.

**Areas for Improvement**
1. **Depth of Analysis**
   - The discussion stays at a surface level; it lists pros and cons without providing concrete evidence, case studies, or data.
   - Little critical evaluation of trade‑offs (e.g., how bias interacts with job displacement) or policy implications.
   - Recommendation: Incorporate specific examples, statistics, or theoretical frameworks to move from description to analysis.

2. **Language & Style**
   - Repetitive phrasing (“AI … impacts …”, “AI … influences …”) reduces stylistic variety.
   - Minimal 